[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/40_linear_regression.ipynb)

# 🟡 Medium: Linear Regression

Implement **linear regression** using three different approaches — all in pure PyTorch.

Given data `X` of shape `(N, D)` and targets `y` of shape `(N,)`, find weight `w` of shape `(D,)` and bias `b` (scalar) such that:

$$\hat{y} = Xw + b$$

### Signature
```python
class LinearRegression:
    def closed_form(self, X: Tensor, y: Tensor) -> tuple[Tensor, Tensor]: ...
    def gradient_descent(self, X: Tensor, y: Tensor, lr=0.01, steps=1000) -> tuple[Tensor, Tensor]: ...
    def nn_linear(self, X: Tensor, y: Tensor, lr=0.01, steps=1000) -> tuple[Tensor, Tensor]: ...
```

All methods return `(w, b)` where `w` has shape `(D,)` and `b` has shape `()`.

### Method 1 — Closed-Form (Normal Equation)
Augment X with a ones column, then solve:

$$\theta = (X_{aug}^T X_{aug})^{-1} X_{aug}^T y$$

Or use `torch.linalg.lstsq` / `torch.linalg.solve`.

### Method 2 — Gradient Descent from Scratch
Initialize `w` and `b` to zeros. Repeat for `steps` iterations:
```
pred = X @ w + b
error = pred - y
grad_w = (2/N) * X^T @ error
grad_b = (2/N) * error.sum()
w -= lr * grad_w
b -= lr * grad_b
```

### Method 3 — PyTorch nn.Linear
Create `nn.Linear(D, 1)`, use `nn.MSELoss` and an optimizer (e.g., `torch.optim.SGD`).
After training, extract `w` and `b` from the layer.

### Rules
- All inputs and outputs must be **PyTorch tensors**
- Do **NOT** use numpy or sklearn
- `closed_form` must not use iterative optimization
- `gradient_descent` must manually compute gradients (no `autograd`)
- `nn_linear` should use `torch.nn.Linear` and `loss.backward()`

In [52]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.4 MB/s eta 0:00:00


In [72]:
import torch
import torch.nn as nn

In [73]:
# ✏️ YOUR IMPLEMENTATION HERE

class LinearRegression:
    def closed_form(self, X: torch.Tensor, y: torch.Tensor):
        """Normal equation: w = (X^T X)^{-1} X^T y"""
        ones = torch.ones(X.shape[0], 1)
        X_b = torch.cat([ones, X], dim=1)
        w_ = torch.linalg.lstsq(X_b, y).solution
        return (w_[1:], w_[0])

    def gradient_descent(self, X: torch.Tensor, y: torch.Tensor,
                         lr: float = 0.01, steps: int = 1000):
        N = X.shape[0]
        w, b = torch.rand(X.shape[-1]), torch.rand(1)
        for _ in range(steps):
          pred = X @ w + b
          error = pred - y
          grad_w = (2 / N) * X.T @ error
          grad_b = (2 / N) * error.sum()
          w -= lr * grad_w
          b -= lr * grad_b
        return (w, b)

    def nn_linear(self, X: torch.Tensor, y: torch.Tensor,
                  lr: float = 0.01, steps: int = 1000):
        """Train nn.Linear with autograd"""
        n, d = X.shape
        linear = nn.Linear(d, 1)
        op = torch.optim.SGD(linear.parameters(), lr=lr)
        loss_fn = nn.MSELoss()
        y = y.view(-1, 1)
        for _ in range(steps):
          op.zero_grad()
          y_ = linear(X)
          loss = loss_fn(y_, y)
          loss.backward()
          op.step()
        w = linear.weight.detach().squeeze() # 从 (1, d) 压缩成长度为 d 的一维向量
        b = linear.bias.detach().squeeze()   # 变成一个标量

        return w, b

In [74]:
# 🧪 Debug
torch.manual_seed(42)
X = torch.randn(100, 3)
true_w = torch.tensor([2.0, -1.0, 0.5])
y = X @ true_w + 3.0

model = LinearRegression()

w_cf, b_cf = model.closed_form(X, y)
print(f"Closed-form:  w={w_cf}, b={b_cf.item():.4f}")

w_gd, b_gd = model.gradient_descent(X, y, lr=0.05, steps=2000)
print(f"Grad descent: w={w_gd}, b={b_gd.item():.4f}")

w_nn, b_nn = model.nn_linear(X, y, lr=0.05, steps=2000)
print(f"nn.Linear:    w={w_nn}, b={b_nn.item():.4f}")

print(f"\nTrue:         w={true_w}, b=3.0")

Closed-form:  w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0000
Grad descent: w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0000
nn.Linear:    w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0000

True:         w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0


In [66]:
# ✅ SUBMIT
from torch_judge import check
check("linear_regression")


🧪 Testing: Linear Regression (Medium)
──────────────────────────────────────────────────
  ✅ [1/6] Closed-form returns correct shapes (4.7ms)
  ✅ [2/6] Closed-form finds correct weights (4.5ms)
  ✅ [3/6] Gradient descent converges (133.7ms)
  ✅ [4/6] nn.Linear approach works (706.9ms)
  ✅ [5/6] All three methods agree (1207.5ms)
  ✅ [6/6] Closed-form uses no autograd (0.9ms)
──────────────────────────────────────────────────
  🎉 All 6 tests passed! (2058.2ms total)
  Progress saved. Run status() to see your dashboard.



In [75]:
y

tensor([ 5.8169e+00, -2.5067e+00,  4.1425e+00,  5.9881e+00,  1.7192e+00,
         2.8028e+00,  1.1866e+00,  5.1961e+00,  4.5671e+00,  5.9220e+00,
         9.4465e-01,  2.3396e+00,  4.2151e+00,  3.5062e+00,  1.2045e+00,
         8.2363e+00,  1.8695e+00,  5.1353e+00,  1.4064e+00,  3.4733e+00,
         3.6752e-01,  4.7502e+00,  7.0863e+00,  4.2989e+00,  2.1601e+00,
         2.9567e+00, -6.5656e-01,  3.3705e+00,  4.1184e+00,  2.1185e+00,
        -5.6398e-01,  2.5153e-01, -2.1148e+00,  2.7082e+00,  5.4914e+00,
         4.1900e+00,  2.2560e+00,  3.7987e+00,  1.6877e+00,  3.8353e+00,
         2.8001e+00,  3.6034e+00,  4.9941e+00,  5.8952e+00,  1.4199e+00,
         3.0198e+00, -1.7247e+00,  6.0712e+00,  9.0579e-01,  5.5266e+00,
         1.6045e+00,  6.2033e+00, -2.1404e+00,  6.6395e+00,  3.5391e+00,
         8.1653e-01, -9.1280e-01,  5.0298e+00,  2.5424e+00,  3.9431e+00,
         2.1591e+00,  1.7229e+00, -4.6970e-01,  1.5391e+00,  7.9516e+00,
         3.6803e+00,  2.8669e+00,  7.2788e+00,  2.9

In [81]:
linear = nn.Linear(X.shape[1], 1)